# Step 11b — LaTeX tables for IRR

> Render the IRR report into the paper-ready `.tex` fragments under `data/tables/` (matching the layout used by Section 'Study Steps and Procedures').


## Inputs
| Path | Format | Source |
|---|---|---|
| `data/step11_report.json` | JSON | Step 11 |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/tables/irr_summary.tex` | LaTeX fragment | Paper |
| `data/tables/irr_confusion_overall.tex` | LaTeX fragment | Paper |
| `data/step11b_report.json` | JSON (input + output SHA-256 hashes) | Integrity audit |


In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 11b — starting: emit IRR LaTeX tables...", flush=True)
r"""Step 11b: Generate LaTeX tables from the step 11 IRR report.

Reads the JSON report produced by step 11 and emits the LaTeX table files
ready for \input{} in the paper. (The former ground-truth-accuracy / label-distribution
tables were retired together with the step 11a cluster.)

Inputs:
  data/step11_report.json          IRR analysis report (step 11)

Outputs:
  data/tables/irr_summary.tex           Per-pair Kappa + concordance
  data/tables/irr_confusion_overall.tex Overall A-vs-B confusion matrix
  data/step11b_report.json              Step report with output hashes
"""

import argparse
import json
import sys
from pathlib import Path

REPORT_11_PATH = Path("data/step11_report.json")
TABLES_DIR = Path("data/tables")
REPORT_PATH = Path("data/step11b_report.json")

def run(
    report11_path: Path = REPORT_11_PATH,
    tables_dir: Path = TABLES_DIR,
    report_path: Path = REPORT_PATH,
) -> dict[str, Path]:
    try:
        sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
    except NameError:
        pass  # Jupyter: __file__ undefined; dataset_utils on path via wheel install
    from dataset_utils.latex import (
        write_irr_confusion_overall,
        write_irr_summary,
    )

    if not report11_path.exists():
        msg = f"step 11b: missing {report11_path} — run step 11 first."
        raise FileNotFoundError(msg)

    with report11_path.open(encoding="utf-8") as f:
        r11 = json.load(f)

    out: dict[str, Path] = {}

    p = tables_dir / "irr_summary.tex"
    write_irr_summary(r11, p)
    out["irr_summary"] = p

    p = tables_dir / "irr_confusion_overall.tex"
    write_irr_confusion_overall(r11, p)
    out["irr_confusion_overall"] = p

    from dataset_utils.hashing import compute_file_hash

    report = {
        "input_report11_sha256": compute_file_hash(report11_path),
        "outputs": {name: {"path": str(p), "sha256": compute_file_hash(p)} for name, p in out.items()},
    }
    report_path.parent.mkdir(parents=True, exist_ok=True)
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    return out

def _parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Generate IRR LaTeX tables from step 11.")
    p.add_argument("--report11", type=Path, default=REPORT_11_PATH)
    p.add_argument("--tables-dir", type=Path, default=TABLES_DIR)
    p.add_argument("--report", type=Path, default=REPORT_PATH)
    return p.parse_args()

def main() -> None:
    args = _parse_args()
    outputs = run(args.report11, args.tables_dir, args.report)
    print("LaTeX tables generated.")
    for name, path in outputs.items():
        print(f"  {name:<30} {path}")
    print(f"  {'report':<30} {args.report}")

main()

Step 11b — starting: emit IRR LaTeX tables...
LaTeX tables generated.
  irr_summary                    data/tables/irr_summary.tex
  irr_confusion_overall          data/tables/irr_confusion_overall.tex
  report                         data/step11b_report.json


---

**Next step:** [`12_human_validated_ground_truth.ipynb`](./12_human_validated_ground_truth.ipynb)
